In [1]:
import pandas as pd


In [2]:
df = pd.read_parquet("C:\\Users\\Mansi\\OneDrive\\Desktop\\Uber\\yellow_tripdata_2026-01.parquet")
df.to_csv(
    r"C:\\Users\\Mansi\\Downloads\\yellow_tripdata_2026-01.csv",
    index=False
)
print("converted parquet to csv")

converted parquet to csv


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [4]:
pd.set_option('display.max_columns',None)
df.head

<bound method NDFrame.head of          VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0               2  2026-01-01 00:54:04   2026-01-01 00:59:37              1.0   
1               1  2026-01-01 00:34:04   2026-01-01 00:39:47              0.0   
2               1  2026-01-01 00:57:06   2026-01-01 01:05:59              0.0   
3               2  2026-01-01 00:15:22   2026-01-01 00:58:10              4.0   
4               2  2026-01-01 00:27:13   2026-01-01 00:40:43              0.0   
...           ...                  ...                   ...              ...   
3724884         2  2026-01-31 23:26:00   2026-01-31 23:39:16              NaN   
3724885         2  2026-01-31 23:33:53   2026-01-31 23:34:07              NaN   
3724886         2  2026-01-31 23:40:23   2026-01-31 23:56:10              NaN   
3724887         2  2026-01-31 23:10:21   2026-01-31 23:20:00              NaN   
3724888         2  2026-01-31 23:43:12   2026-01-31 23:57:45              NaN  

In [5]:
print("dataset shape")
df.shape

dataset shape


(3724889, 20)

In [6]:
print("missing values (nulls/nans)")
df.isnull().sum()

missing values (nulls/nans)


VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [7]:
print("duplicates present")
df.duplicated().sum()

duplicates present


np.int64(0)

In [8]:
df = df.dropna()
df = df.reset_index(drop=True)

In [9]:
df.isnull().sum()

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
Airport_fee              0
cbd_congestion_fee       0
dtype: int64

In [10]:
df['trip_id'] = df.index

In [11]:
payment_dim = df[['payment_type']].copy()
payment_dim = payment_dim.drop_duplicates()
payment_dim = payment_dim.reset_index(drop=True)
payment_dim.insert(0,"payment_key",payment_dim.index+1)

payment_mapping = {
    1:"Credit card",
    2:"Cash",
    3:"No charge",
    4:"Dispute",
    5:"Unknown",
    6:"Voided trip"
}

payment_dim["payment_name"] = payment_dim["payment_type"].map(payment_mapping)


In [12]:
datetime_dim = df[['tpep_pickup_datetime']].copy()
datetime_dim = datetime_dim.drop_duplicates()
datetime_dim = datetime_dim.reset_index(drop=True)
datetime_dim.insert(0,"datetime_key",datetime_dim.index+1)

datetime_dim["tpep_pickup_datetime"] = pd.to_datetime(datetime_dim["tpep_pickup_datetime"])
datetime_dim["year"] = datetime_dim["tpep_pickup_datetime"].dt.year
datetime_dim["month"] = datetime_dim["tpep_pickup_datetime"].dt.month
datetime_dim["day"] = datetime_dim["tpep_pickup_datetime"].dt.day

datetime_dim["hour"] = datetime_dim["tpep_pickup_datetime"].dt.hour
datetime_dim["minute"] = datetime_dim["tpep_pickup_datetime"].dt.minute
datetime_dim["weekday"] = datetime_dim["tpep_pickup_datetime"].dt.day_name()
datetime_dim["quarter"] = datetime_dim["tpep_pickup_datetime"].dt.quarter



In [13]:
location_ids = pd.concat([
    df["PULocationID"],
    df["DOLocationID"]
]).drop_duplicates().reset_index(drop=True)

location_dim = pd.DataFrame({
    "location_id": location_ids
})

location_dim.insert(
    0,
    "location_key",
    location_dim.index + 1
)

location_dim.head()

,location_key,location_id
0,1,239
1,2,163
2,3,43
3,4,142
4,5,88


In [14]:
vendor_dim = df[['VendorID']].copy()
vendor_dim = vendor_dim.drop_duplicates()
vendor_dim = vendor_dim.reset_index(drop=True)
vendor_dim.insert(
    0,
    "vendor_key",
    vendor_dim.index + 1
)
vendor_mapping = {
    1: "Creative Mobile Technologies",
    2: "VeriFone Inc."
}

vendor_dim["vendor_name"] = vendor_dim["VendorID"].map(vendor_mapping)
vendor_dim

,vendor_key,VendorID,vendor_name
0,1,2,VeriFone Inc.
1,2,1,Creative Mobile Technologies
2,3,7,NaN


In [15]:
rate_dim = df[['RatecodeID']].copy()
rate_dim = rate_dim.drop_duplicates()
rate_dim = rate_dim.reset_index(drop=True)
rate_dim.insert(
    0,
    "rate_code_key",
    rate_dim.index + 1
)
rate_mapping = {
    1: "Standard Rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau/Westchester",
    5: "Negotiated Fare",
    6: "Group Ride"
}
rate_dim["rate_name"] = rate_dim["RatecodeID"].map(rate_mapping)

rate_dim

,rate_code_key,RatecodeID,rate_name
0,1,1.0,Standard Rate
1,2,4.0,Nassau/Westchester
2,3,2.0,JFK
3,4,5.0,Negotiated Fare
4,5,99.0,NaN
5,6,3.0,Newark
6,7,6.0,Group Ride


In [16]:
fact_table = df.copy()

In [17]:
fact_table = fact_table.merge(vendor_dim, on="VendorID", how="left")

In [18]:
fact_table = fact_table.merge(datetime_dim[["datetime_key", "tpep_pickup_datetime"]], on="tpep_pickup_datetime", how="left")

In [19]:
fact_table = fact_table.merge(payment_dim, on="payment_type", how="left")

In [20]:
fact_table = fact_table.merge(location_dim, left_on="PULocationID", right_on="location_id", how="left")
fact_table.rename(columns={"location_key": "pickup_location_key"}, inplace=True)
fact_table.drop(columns=["location_id"], inplace=True)

In [21]:
fact_table = fact_table.merge(location_dim, left_on="DOLocationID", right_on="location_id", how="left")
fact_table.rename(columns={"location_key": "dropoff_location_key"}, inplace=True)
fact_table.drop(columns=["location_id"], inplace=True)

In [22]:
fact_table = fact_table[
    [
        "vendor_key",
        "datetime_key",
        "payment_key",
        "pickup_location_key",
        "dropoff_location_key",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount"
    ]
]


In [23]:
fact_table.head()

,vendor_key,datetime_key,payment_key,pickup_location_key,dropoff_location_key,passenger_count,trip_distance,fare_amount,tip_amount,total_amount
0,1,1,1,1,24,1.0,0.97,7.2,3.66,15.86
1,2,2,2,2,11,0.0,0.90,7.9,0.00,13.65
2,2,3,1,3,9,0.0,1.40,10.7,2.50,18.95
3,1,4,1,4,54,4.0,5.58,38.7,11.11,55.56
4,1,5,1,5,6,0.0,2.16,13.5,3.85,23.10


In [24]:
import pandas_gbq

project_id = "maximal-yew-474913-s0"
dataset_id = "nyc_taxi_dw"

tables = {
    "fact_table": fact_table,
    "vendor_dim": vendor_dim,
    "datetime_dim": datetime_dim,
    "payment_dim": payment_dim,
    "location_dim": location_dim,
    "rate_dim": rate_dim,
}

for table_name, df in tables.items():
    pandas_gbq.to_gbq(
        df,
        destination_table=f"{dataset_id}.{table_name}",
        project_id=project_id,
        if_exists="replace"
    )
    print(f"Loaded {table_name} ({len(df)} rows)")

Loaded fact_table (2636831 rows)
Loaded vendor_dim (3 rows)
Loaded datetime_dim (1452956 rows)
Loaded payment_dim (4 rows)
Loaded location_dim (262 rows)
Loaded rate_dim (7 rows)
